## YOLOv8s Baseline - v1 Enhanced Dataset

Trains the YOLOv8s baseline detector on the v1 enhanced dataset (photometric low-light augmentation: gamma / CLAHE / brightness-contrast, train-only). Same training config as the raw-data baseline for a fair comparison: 100 epochs, imgsz 640, batch 16, patience 20, seed 42.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for root, dirs, files in os.walk('/kaggle/input'):
    if "data.yaml" in files:
        print(os.path.join(root, "data.yaml"))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# Install Ultralytics
!pip install ultralytics

In [ ]:
# prepare dataset yaml for kaggle
import shutil
import yaml

# Original (read-only)
yaml_path = "/kaggle/input/datasets/jharshin/v1-enhanced-dataset/enhanced_yolo_dataset/data.yaml"

# Copy location (writable)
new_yaml_path = "/kaggle/working/data.yaml"

# Copy yaml
shutil.copy(yaml_path, new_yaml_path)

# Load copied yaml
with open(new_yaml_path, "r") as f:
    data = yaml.safe_load(f)

# Update paths for Kaggle
data["path"] = "/kaggle/input/datasets/jharshin/v1-enhanced-dataset/enhanced_yolo_dataset"
data["train"] = "images/train"
data["val"] = "images/val"
data["test"] = "images/test"

# Save modified yaml
with open(new_yaml_path, "w") as f:
    yaml.dump(data, f)

print("Updated data.yaml saved:", new_yaml_path)

### Train YOLOv8s baseline (v1 enhanced dataset)

In [ ]:
# Yolov8s model on v1 enhanced dataset
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    seed=42,  # fixed seed for a reproducible, comparable run
    patience=20,  # stop early if val mAP hasn't improved in 20 epochs
    project="/kaggle/working/ntlnp_experiments",
    name="baseline_yolov8s_v1data")

### Evaluate best checkpoint

In [ ]:
# assining the best model
from ultralytics import YOLO
best_model = YOLO("/kaggle/working/ntlnp_experiments/baseline_yolov8s_v1data/weights/best.pt")

In [ ]:
# evaluation on the validation set
val_results = best_model.val(
    data="/kaggle/working/data.yaml",
    split="val"
)

print("Validation mAP50:", val_results.box.map50)
print("Validation mAP50-95:", val_results.box.map)
print("Validation Precision:", val_results.box.mp)
print("Validation Recall:", val_results.box.mr)

In [ ]:
# Evaluation on the test set
test_results = best_model.val(
    data="/kaggle/working/data.yaml",
    split="test"
)

print("Test mAP50:", test_results.box.map50)
print("Test mAP50-95:", test_results.box.map)
print("Test Precision:", test_results.box.mp)
print("Test Recall:", test_results.box.mr)

In [ ]:
# Predictions on test set
prediction_results = best_model.predict(
    source="/kaggle/input/datasets/jharshin/v1-enhanced-dataset/enhanced_yolo_dataset/images/test",
    conf=0.25,
    save=True
)

In [ ]:
# print per-class AP
names = best_model.names
for i, name in names.items():
    print(
        f"{name:<15} "
        f"AP50: {test_results.box.ap50[i]:.4f} "
        f"AP50-95: {test_results.box.ap[i]:.4f}"
    )

### Compile & save metrics

In [ ]:
# Evaluation results
metrics = {
    "Model": "baseline_yolov8s_v1data",
    "Validation mAP50": val_results.box.map50,
    "Validation mAP50-95": val_results.box.map,
    "Validation Precision": val_results.box.mp,
    "Validation Recall": val_results.box.mr,
    "Test mAP50": test_results.box.map50,
    "Test mAP50-95": test_results.box.map,
    "Test Precision": test_results.box.mp,
    "Test Recall": test_results.box.mr,
}

print(metrics)

In [ ]:
# saving the metrics as CSV
import pandas as pd
import os

results_file = "/kaggle/working/ntlnp_experiments/experiment_results_v1data.csv"

df = pd.DataFrame([metrics])

if os.path.exists(results_file):
    old = pd.read_csv(results_file)
    df = pd.concat([old, df], ignore_index=True)

df.to_csv(results_file, index=False)

print(df)

In [ ]:
# package results for download
!zip -r kaggle_working_v1.zip /kaggle/working

### Dataset integrity check

In [ ]:
# dataset integrity check
from pathlib import Path

for split in ["train", "val"]:
    img_path = Path("/kaggle/input/datasets/jharshin/v1-enhanced-dataset/enhanced_yolo_dataset/images") / split
    label_path = Path("/kaggle/input/datasets/jharshin/v1-enhanced-dataset/enhanced_yolo_dataset/labels") / split

    images = {x.stem for x in img_path.iterdir()}
    labels = {x.stem for x in label_path.iterdir()}

    print(split)
    print("Images without labels:", images-labels)
    print("Labels without images:", labels-images)